<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/modelTrain_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Improved Agile Priority Classifier
## Upgrade: BERT-base + Class Weights + Longer Sequences + Better Training

**Key improvements over v1 (TinyBERT):**
- ✅ `bert-base-uncased` — much stronger backbone (12 layers vs 4)
- ✅ Class-weighted loss — fixes Medium class confusion
- ✅ `max_length=256` — captures full requirement text
- ✅ Focal loss option — penalises easy examples less
- ✅ Warmup + cosine LR schedule — smoother convergence
- ✅ More epochs (8) with early stopping — avoids underfitting
- ✅ Label smoothing — improves calibration
- ✅ Gradient clipping — stable training


In [1]:
#═══════════════════════════════════════════
# STEP 1: Install required packages
#═══════════════════════════════════════════
import os
os.environ["WANDB_DISABLED"] = "true"

!pip uninstall transformers accelerate peft -y -q
!pip install transformers==4.40.0 accelerate==0.27.2 peft==0.9.0 -q

print("✅ Installation complete!")
print("   → Runtime → Restart Session → Ok")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
✅ Installation complete!
   → Runtime → Restart Session → Ok


In [2]:
#═══════════════════════════════════════════
# STEP 2: Verify GPU + versions
#═══════════════════════════════════════════
import torch, transformers, accelerate

print(f"GPU Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name      : {torch.cuda.get_device_name(0)}")

print(f"transformers  : {transformers.__version__}")
print(f"accelerate    : {accelerate.__version__}")

if not torch.cuda.is_available():
    print("\n⚠️  No GPU! Runtime → Change Runtime Type → T4 GPU")
else:
    print("\n✅ Ready!")


GPU Available : True
GPU Name      : Tesla T4
transformers  : 4.40.0
accelerate    : 0.27.2

✅ Ready!


In [3]:
#═══════════════════════════════════════════
# STEP 3: Upload CSV files
#═══════════════════════════════════════════
import os, pandas as pd
from google.colab import files

os.environ["WANDB_DISABLED"] = "true"

print("Upload 4 files:")
print("  1. train_combined.csv")
print("  2. val_combined.csv")
print("  3. test_combined.csv")
print("  4. unlabeled_dataset.csv")
print("")

uploaded = files.upload()

train_df     = pd.read_csv("train_combined.csv")
val_df       = pd.read_csv("val_combined.csv")
test_df      = pd.read_csv("test_combined.csv")
df_unlabeled = pd.read_csv("unlabeled_dataset.csv",
                           header=None, names=['text', 'source'])

print(f"\n✅ All files loaded!")
print(f"   train_df     : {len(train_df):,} rows")
print(f"   val_df       : {len(val_df):,} rows")
print(f"   test_df      : {len(test_df):,} rows")
print(f"   df_unlabeled : {len(df_unlabeled):,} requirements")
print(f"\nPriority Distribution (train):")
print(train_df['priority'].value_counts())


Upload 4 files:
  1. train_combined.csv
  2. val_combined.csv
  3. test_combined.csv
  4. unlabeled_dataset.csv



Saving unlabeled_dataset.csv to unlabeled_dataset.csv
Saving test_combined.csv to test_combined.csv
Saving val_combined.csv to val_combined.csv
Saving train_combined.csv to train_combined.csv

✅ All files loaded!
   train_df     : 23,274 rows
   val_df       : 2,909 rows
   test_df      : 2,910 rows
   df_unlabeled : 98 requirements

Priority Distribution (train):
priority
Medium    9430
High      8550
Low       5294
Name: count, dtype: int64


In [4]:
#═══════════════════════════════════════════
# STEP 4: Compute class weights
# Fixes Medium class being under-learned
#═══════════════════════════════════════════
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch

labels_array = train_df['priority_id'].values
classes      = np.array([0, 1, 2])

weights = compute_class_weight(
    class_weight = 'balanced',
    classes      = classes,
    y            = labels_array
)

class_weights = torch.tensor(weights, dtype=torch.float)
print("Class weights (balanced):")
print(f"  High   (0): {weights[0]:.4f}")
print(f"  Medium (1): {weights[1]:.4f}")
print(f"  Low    (2): {weights[2]:.4f}")
print("\n✅ Higher weight = model focuses more on that class")


Class weights (balanced):
  High   (0): 0.9074
  Medium (1): 0.8227
  Low    (2): 1.4654

✅ Higher weight = model focuses more on that class


In [5]:
#═══════════════════════════════════════════
# STEP 5: Load BERT-base-uncased
# Upgrade from TinyBERT → full BERT
# Much stronger: 12 layers vs 4, 768 dims vs 312
#═══════════════════════════════════════════
import torch
from transformers import BertTokenizer, BertForSequenceClassification

model_name = "bert-base-uncased"

print(f"Loading {model_name}...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model     = BertForSequenceClassification.from_pretrained(
                model_name,
                num_labels         = 3,   # High=0  Medium=1  Low=2
                hidden_dropout_prob = 0.2,   # slightly higher dropout
            )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"\n✅ {model_name} loaded!")
print(f"   Device     : {device}")
print(f"   Parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"   Labels     : High(0)  Medium(1)  Low(2)")


Loading bert-base-uncased...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ bert-base-uncased loaded!
   Device     : cuda
   Parameters : 109,484,547
   Labels     : High(0)  Medium(1)  Low(2)


In [6]:
#═══════════════════════════════════════════
# STEP 6: Dataset class + Weighted Trainer
# max_length=256 captures full requirement text
# Custom trainer applies class-weighted loss
#═══════════════════════════════════════════
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

MAX_LENGTH = 256   # ↑ was 128 — captures more context

class RequirementsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=MAX_LENGTH):
        self.texts     = dataframe['text'].tolist()
        self.labels    = dataframe['priority_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            truncation     = True,
            padding        = 'max_length',
            max_length     = self.max_len,
            return_tensors = 'pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels'        : torch.tensor(
                                  self.labels[index], dtype=torch.long
                              )
        }

train_dataset = RequirementsDataset(train_df, tokenizer)
val_dataset   = RequirementsDataset(val_df,   tokenizer)
test_dataset  = RequirementsDataset(test_df,  tokenizer)

# ── Custom Trainer with class-weighted cross-entropy ──
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
        else:
            loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Metrics (add macro f1) ──
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds          = np.argmax(logits, axis=1)
    return {
        'accuracy'     : round(accuracy_score(labels, preds), 4),
        'f1_weighted'  : round(f1_score(labels, preds, average='weighted'), 4),
        'f1_macro'     : round(f1_score(labels, preds, average='macro'), 4),
    }

print(f"✅ Datasets and WeightedTrainer ready!")
print(f"   Training   : {len(train_dataset):,} samples")
print(f"   Validation : {len(val_dataset):,} samples")
print(f"   Test       : {len(test_dataset):,} samples")
print(f"   Max length : {MAX_LENGTH} tokens")


✅ Datasets and WeightedTrainer ready!
   Training   : 23,274 samples
   Validation : 2,909 samples
   Test       : 2,910 samples
   Max length : 256 tokens


In [7]:
#═══════════════════════════════════════════
# STEP 7: Configure improved training
# Key upgrades vs v1:
#   epochs        5  → 8  (more training time)
#   batch_size   16  → 16 (same; increase to 32 if GPU allows)
#   learning_rate 2e-5 → 3e-5 (BERT-base benefits from slightly higher LR)
#   warmup_ratio   0  → 0.1  (warmup prevents early instability)
#   weight_decay  0.01 → 0.01 (keep regularisation)
#   lr_scheduler  linear → cosine (smoother decay)
#   max_grad_norm  —  → 1.0  (gradient clipping)
#   label_smoothing → applied inside WeightedTrainer
#═══════════════════════════════════════════
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir                  = './bert-priority-v2',

    # ── Training schedule ──
    num_train_epochs            = 8,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,

    # ── Learning rate ──
    learning_rate               = 3e-5,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.1,        # 10% of steps = warmup

    # ── Regularisation ──
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,        # gradient clipping

    # ── Evaluation & saving ──
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',   # ← optimise macro F1
    greater_is_better           = True,

    # ── Logging ──
    logging_steps               = 50,
    report_to                   = 'none',
    fp16                        = True,         # mixed precision (GPU)
)

# ── Instantiate trainer ──
trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    class_weights   = class_weights,
)

print("✅ Training configuration ready!")
print("   Model          : bert-base-uncased")
print("   Epochs         : 8")
print("   LR             : 3e-5 (cosine + warmup)")
print("   Loss           : Weighted CrossEntropy + label smoothing 0.1")
print("   Best model by  : macro F1")
print("   Grad clip      : 1.0")
print("   Mixed precision: fp16")


✅ Training configuration ready!
   Model          : bert-base-uncased
   Epochs         : 8
   LR             : 3e-5 (cosine + warmup)
   Loss           : Weighted CrossEntropy + label smoothing 0.1
   Best model by  : macro F1
   Grad clip      : 1.0
   Mixed precision: fp16


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [8]:
#═══════════════════════════════════════════
# STEP 8: Fine-tune BERT-base
# ⏱  ~35-50 min on T4 GPU
#═══════════════════════════════════════════
print("=" * 55)
print("      STARTING BERT-BASE FINE-TUNING")
print("=" * 55)
print(f"   Model    : bert-base-uncased")
print(f"   Task     : Agile Requirement Prioritization")
print(f"   Classes  : High | Medium | Low")
print(f"   Samples  : {len(train_dataset):,}")
print(f"   Epochs   : 8")
print(f"   Time     : ~35-50 minutes on T4")
print("=" * 55)
print("\nTraining in progress...\n")

trainer.train()

print("\n✅ Training Complete!")


      STARTING BERT-BASE FINE-TUNING
   Model    : bert-base-uncased
   Task     : Agile Requirement Prioritization
   Classes  : High | Medium | Low
   Samples  : 23,274
   Epochs   : 8
   Time     : ~35-50 minutes on T4

Training in progress...



Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.718600,0.728805,0.736000,0.734600,0.744800
2,0.646200,0.678551,0.754900,0.755100,0.763900
3,0.635600,0.693978,0.755200,0.752800,0.757800
4,0.558700,0.748856,0.725700,0.723200,0.728500
5,0.531400,0.748756,0.757300,0.757000,0.764100
6,0.460400,0.805771,0.747300,0.746800,0.753500


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.718600,0.728805,0.736000,0.734600,0.744800
2,0.646200,0.678551,0.754900,0.755100,0.763900
3,0.635600,0.693978,0.755200,0.752800,0.757800
4,0.558700,0.748856,0.725700,0.723200,0.728500
5,0.531400,0.748756,0.757300,0.757000,0.764100
6,0.460400,0.805771,0.747300,0.746800,0.753500
7,0.431100,0.859244,0.742200,0.741200,0.746900
8,0.413500,0.868409,0.744200,0.743500,0.750000



✅ Training Complete!


In [9]:
#═══════════════════════════════════════════
# STEP 9: Save model — do immediately!
#═══════════════════════════════════════════
import shutil
from google.colab import drive

drive.mount('/content/drive')

model.save_pretrained("./bert-priority-v2")
tokenizer.save_pretrained("./bert-priority-v2")
print("✅ Model saved locally!")

shutil.copytree(
    "./bert-priority-v2",
    "/content/drive/MyDrive/bert-priority-v2",
    dirs_exist_ok=True
)
print("✅ Model saved to Google Drive!")
print("   Location: MyDrive/bert-priority-v2")


Mounted at /content/drive
✅ Model saved locally!
✅ Model saved to Google Drive!
   Location: MyDrive/bert-priority-v2


In [10]:
#═══════════════════════════════════════════
# STEP 10: Evaluate on test set
# Target: accuracy > 0.82, macro avg > 0.82
#═══════════════════════════════════════════
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['priority_id'].tolist()

print("=" * 60)
print("         FINAL MODEL EVALUATION")
print("=" * 60)
report = classification_report(
    true_labels,
    pred_labels,
    target_names=['High', 'Medium', 'Low'],
    digits=4
)
print(report)

cm    = confusion_matrix(true_labels, pred_labels)
cm_df = pd.DataFrame(
    cm,
    index   = ['Actual High', 'Actual Medium', 'Actual Low'],
    columns = ['Pred High',   'Pred Medium',   'Pred Low']
)
print("Confusion Matrix:")
print(cm_df)

# ── Quick comparison ──
print("\n" + "=" * 60)
print("  IMPROVEMENT SUMMARY vs TinyBERT v1")
print("=" * 60)
from sklearn.metrics import accuracy_score, f1_score
new_acc   = accuracy_score(true_labels, pred_labels)
new_macro = f1_score(true_labels, pred_labels, average='macro')
print(f"  Accuracy  : 0.76 (v1)  →  {new_acc:.4f} (v2)")
print(f"  Macro F1  : 0.77 (v1)  →  {new_macro:.4f} (v2)")
print("=" * 60)


         FINAL MODEL EVALUATION
              precision    recall  f1-score   support

        High     0.7377    0.7287    0.7332      1069
      Medium     0.7502    0.7413    0.7457      1179
         Low     0.7925    0.8248    0.8083       662

    accuracy                         0.7557      2910
   macro avg     0.7601    0.7649    0.7624      2910
weighted avg     0.7552    0.7557    0.7554      2910

Confusion Matrix:
               Pred High  Pred Medium  Pred Low
Actual High          779          227        63
Actual Medium        225          874        80
Actual Low            52           64       546

  IMPROVEMENT SUMMARY vs TinyBERT v1
  Accuracy  : 0.76 (v1)  →  0.7557 (v2)
  Macro F1  : 0.77 (v1)  →  0.7624 (v2)


In [11]:
#═══════════════════════════════════════════
# STEP 11: Error analysis
# Understand where the model still struggles
#═══════════════════════════════════════════
import pandas as pd

results_df = test_df.copy()
results_df['pred_id']   = pred_labels
results_df['true_id']   = true_labels
results_df['correct']   = results_df['pred_id'] == results_df['true_id']

label_map = {0: 'High', 1: 'Medium', 2: 'Low'}
results_df['pred_label'] = results_df['pred_id'].map(label_map)
results_df['true_label'] = results_df['true_id'].map(label_map)

# Show worst errors (Medium predicted as High — the main confusion)
errors = results_df[~results_df['correct']]
print(f"Total errors: {len(errors)} / {len(results_df)} ({len(errors)/len(results_df)*100:.1f}%)")
print("\nError breakdown (true → predicted):")
print(errors.groupby(['true_label', 'pred_label']).size().sort_values(ascending=False).to_string())

print("\n--- Sample Medium→High misclassifications ---")
med_high = errors[
    (errors['true_label']=='Medium') & (errors['pred_label']=='High')
].head(5)
for _, row in med_high.iterrows():
    print(f"  [{row['true_label']}→{row['pred_label']}] {row['text'][:80]}...")


Total errors: 711 / 2910 (24.4%)

Error breakdown (true → predicted):
true_label  pred_label
High        Medium        227
Medium      High          225
            Low            80
Low         Medium         64
High        Low            63
Low         High           52

--- Sample Medium→High misclassifications ---
  [Medium→High] Migrate component examples and docs to embedded GitLab UI stories...
  [Medium→High] BE: dev-branches should import dev UserPoolId...
  [Medium→High] Linter runs forever...
  [Medium→High] Social Compass Module...
  [Medium→High] Add a title to video and image entities...


In [12]:
#═══════════════════════════════════════════
# STEP 12: Rank all unlabeled requirements
#═══════════════════════════════════════════
import torch

def predict_priority(text):
    inputs = tokenizer(
        text,
        return_tensors = 'pt',
        truncation     = True,
        padding        = True,
        max_length     = 256
    )
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs      = torch.softmax(outputs.logits, dim=1)[0]
    pred_id    = probs.argmax().item()
    confidence = round(probs[pred_id].item() * 100, 2)
    priority_map = {0: 'High', 1: 'Medium', 2: 'Low'}
    return priority_map[pred_id], confidence, probs.tolist()

print("Ranking all requirements...")
results = []

for i, row in df_unlabeled.iterrows():
    priority, confidence, probs = predict_priority(row['text'])
    results.append({
        'text'        : row['text'],
        'source'      : row['source'],
        'priority'    : priority,
        'confidence'  : confidence,
        'prob_high'   : round(probs[0] * 100, 2),
        'prob_medium' : round(probs[1] * 100, 2),
        'prob_low'    : round(probs[2] * 100, 2),
    })

df_results = pd.DataFrame(results)

priority_order = {'High': 0, 'Medium': 1, 'Low': 2}
df_results['sort_priority']   = df_results['priority'].map(priority_order)
df_results['sort_confidence'] = -df_results['confidence']
df_results = df_results.sort_values(
    ['sort_priority', 'sort_confidence']
).drop(columns=['sort_priority', 'sort_confidence']).reset_index(drop=True)

df_results.index      += 1
df_results.index.name  = 'implement_order'

def suggest_sprint(rank, total):
    pct = rank / total
    if pct <= 0.25:   return 'Sprint 1'
    elif pct <= 0.50: return 'Sprint 2'
    elif pct <= 0.75: return 'Sprint 3'
    else:             return 'Sprint 4'

total = len(df_results)
df_results['sprint'] = [suggest_sprint(i+1, total) for i in range(total)]

print("\n" + "=" * 85)
print("      AGILE REQUIREMENTS — IMPLEMENTATION ORDER")
print("=" * 85)
print(f"{'Rank':<6} {'Priority':<10} {'Sprint':<12} {'Confidence':<13} Requirement")
print("-" * 85)

for idx, row in df_results.iterrows():
    emoji = {'High':'🔴','Medium':'🟡','Low':'🟢'}[row['priority']]
    print(f"{idx:<6} {emoji} {row['priority']:<8} "
          f"{row['sprint']:<12} {row['confidence']}%"
          f"{'':6} {row['text'][:50]}...")

print("\n" + "=" * 85)
print("📊 SUMMARY:")
print(f"   🔴 High   : {len(df_results[df_results['priority']=='High']):>3} → Sprint 1  (Do First)")
print(f"   🟡 Medium : {len(df_results[df_results['priority']=='Medium']):>3} → Sprint 2-3")
print(f"   🟢 Low    : {len(df_results[df_results['priority']=='Low']):>3} → Sprint 4  (Do Last)")
print("=" * 85)


Ranking all requirements...

      AGILE REQUIREMENTS — IMPLEMENTATION ORDER
Rank   Priority   Sprint       Confidence    Requirement
-------------------------------------------------------------------------------------
1      🔴 High     Sprint 1     94.0%       As an Owner, I want to reset the environment to on...
2      🔴 High     Sprint 1     93.51%       As a FABS user, I want to have read-only access to...
3      🔴 High     Sprint 1     92.56%       As an owner, I want to be sure that USAspending on...
4      🔴 High     Sprint 1     63.74%       As an agency user, I want to be confident that the...
5      🔴 High     Sprint 1     40.22%       As a Developer , I want to ensure that attempts to...
6      🟡 Medium   Sprint 1     92.17%       As a agency user, I want to map the FederalActionO...
7      🟡 Medium   Sprint 1     92.05%       As a broker team member, I want to ensure the Brok...
8      🟡 Medium   Sprint 1     91.89%       As a user, I want to generate and validate D Files.

In [ ]:
#═══════════════════════════════════════════
# STEP 13: Download results
#═══════════════════════════════════════════
from google.colab import files

df_results.to_csv("agile_requirements_ranked_v2.csv")
df_results.to_csv("/content/drive/MyDrive/agile_requirements_ranked_v2.csv")
files.download("agile_requirements_ranked_v2.csv")

print("✅ Downloaded: agile_requirements_ranked_v2.csv")
print(f"\n🎯 Complete! {len(df_results)} requirements ranked.")


---
## Optional: Upload a new unlabeled dataset

In [ ]:
#═══════════════════════════════════════════
# OPTIONAL: Upload new unlabeled dataset
# and re-rank with the trained model
#═══════════════════════════════════════════
from google.colab import files as colab_files
import pandas as pd

print("Upload your new unlabeled CSV file:")
new_uploaded = colab_files.upload()

filename = list(new_uploaded.keys())[0]
df_unlabeled = pd.read_csv(filename, header=None, names=['text', 'source'])
print(f"\n✅ {filename} loaded: {len(df_unlabeled):,} requirements")
print("\nNow re-run STEP 12 to rank them.")
